In [ ]:
import torch
from matplotlib import pyplot as plt

from datasets.mnist import MNISTSampler
from models.config import load_config
from models.flow import FlowModel
from training.path import GaussianConditionalProbabilityPath, LinearAlpha, LinearBeta
from training.trainer import FlowTrainer

device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print("device", device)

CONFIG = "configs/mnist.yaml"
cfg = load_config(CONFIG)
flow = FlowModel.from_config(CONFIG).to(device)

xt = torch.randn(
    cfg["test"]["batch_size"],
    cfg["unet"]["in_channels"],
    cfg["test"]["image_size"],
    cfg["test"]["image_size"],
    device=device,
)
t = torch.rand(xt.shape[0], device=device)
print("u(x, t)", tuple(flow(xt, t).shape))

In [ ]:
def make_path(split: str):
    return GaussianConditionalProbabilityPath(
        p_data=MNISTSampler(split=split),
        p_simple_shape=[1, 32, 32],
        alpha=LinearAlpha(),
        beta=LinearBeta(),
    ).to(device)

train_path = make_path("train")
val_path = make_path("val")

trainer = FlowTrainer(path=train_path, model=flow, val_path=val_path)
history = trainer.train(
    num_steps=5000,
    device=device,
    lr=1e-3,
    batch_size=64,
    ckpt_path="checkpoints/mnist_flow.pt",
    checkpoint_every=50,
    val_every=50,
    val_batches=8,
    plot_every=50,
    n_plot_images=10,
    n_plot_steps=10,
    samples_dir="samples",
)

plt.plot(
    torch.arange(1, len(history["train"]) + 1),
    history["train"].numpy(),
    label="train",
)
if "val" in history:
    plt.plot(
        history["val_steps"].numpy(),
        history["val"].numpy(),
        label="val",
    )
plt.xlabel("step")
plt.ylabel("CFM loss")
plt.legend()
plt.show()